### workflow:
1. remove duplicates of reference texts (removed as many as possible, some still exist)
2. establish baseline of performance by running BERTopic with default parameters -> creates too many topics
3. use zero-shot topic modeling to lower number of topics 
4. refine parameters
5. extract finalized topic assignments

In [2]:
import pandas as pd

docs = pd.read_csv('/Users/jamiewong/Documents/school/2025-26/ECS 111/AI-Hallucinations-Detection/data/cleaned_data.csv')['reference'].drop_duplicates()
len(docs)

4544

In [3]:
docs = docs[~docs.str.contains('the most informative paragraph:')] #removes a duplicate with LLM speech in the front
len(docs)

4543

## Unsupervised Topic Modeling

In [4]:
#pre-compute embeddings to speed up BERTopic
from sentence_transformers import SentenceTransformer

docs = docs.to_numpy()
sentence_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings = sentence_model.encode(docs, show_progress_bar=True)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/142 [00:00<?, ?it/s]

In [5]:
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
from bertopic import BERTopic

umap_model = UMAP(random_state=42) #set seed to present stochastic behavior in UMAP

#get english and spanish stop words
languages = ['english', 'spanish']
en_sp_stop_words = set()
for lang in languages:
    en_sp_stop_words.update(stopwords.words(lang))
en_sp_stop_words = list(en_sp_stop_words)

#vectorizer_model = CountVectorizer(stop_words="english")
vectorizer_model = CountVectorizer(stop_words=en_sp_stop_words)

topic_model = BERTopic(language = "multilingual",
                       umap_model = umap_model, #pass seeded model to BERTopic
                       nr_topics = "auto", #merges similar topics
                       vectorizer_model=vectorizer_model) #helps filter stop words
topics, probs = topic_model.fit_transform(docs, embeddings) #fit topic model using docs and pre-computed embeddings

In [6]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1062,-1_data_databricks_women_puede,"[data, databricks, women, puede, use, also, us...",[This get started article walks you through us...
1,0,943,0_data_mongodb_atlas_databricks,"[data, mongodb, atlas, databricks, search, tab...",[Garaudy Etienne joined MongoDB as a Product M...
2,1,480,1_credit_account_bank_financial,"[credit, account, bank, financial, accounts, m...","[However, checking accounts tend to pay low in..."
3,2,361,2_court_law_state_eff,"[court, law, state, eff, apr, supreme, courts,...","[""The single question presented by this record..."
4,3,297,3_noaa_education_program_science,"[noaa, education, program, science, environmen...",[NOAA brings thousands of K-12 teachers and st...
5,4,290,4_ice_storm_water_kilometers,"[ice, storm, water, kilometers, snow, miles, i...",[Tropical Storm Erika was quickly losing steam...
6,5,284,5_pcbs_cancer_health_care,"[pcbs, cancer, health, care, pcb, treatment, s...","[""This is part 4 of PyMongo Monday. Previously..."
7,6,138,6_puede_bacterias_pueden_infecciones,"[puede, bacterias, pueden, infecciones, enferm...",['El lupus es un tipo de enfermedad autoinmune...
8,7,124,7_pmc_nlm_journal_articles,"[pmc, nlm, journal, articles, pubmed, national...","[""PubMed Central® (PMC) is a free full-text ar..."
9,8,64,8_gov_https_official_websites,"[gov, https, official, websites, web, secure, ...",[Secure .gov websites use HTTPS\n\n ...


In [7]:
topic_model.get_params()

{'calculate_probabilities': False,
 'ctfidf_model': ClassTfidfTransformer(),
 'embedding_model': None,
 'hdbscan_model': HDBSCAN(min_cluster_size=10, prediction_data=True),
 'language': 'multilingual',
 'low_memory': False,
 'min_topic_size': 10,
 'n_gram_range': (1, 1),
 'nr_topics': 'auto',
 'representation_model': None,
 'seed_topic_list': None,
 'top_n_words': 10,
 'umap_model': UMAP(n_jobs=1, random_state=42, tqdm_kwds={'bar_format': '{desc}: {percentage:3.0f}%| {bar} {n_fmt}/{total_fmt} [{elapsed}]', 'desc': 'Epochs completed', 'disable': True}),
 'vectorizer_model': CountVectorizer(stop_words=['against', 'estuvierais', 'tuviesen', 'sería',
                             'esa', 'habrías', 'suya', 'son', 'estábamos',
                             'también', 'más', 'nuestro', 'tuviese',
                             'themselves', 'hubiéramos', 'tuviste', 'seríais',
                             'very', 'will', 'desde', 'seréis', 'tenga',
                             "she'll", 'estuviése

In [8]:
hierarchical_topics = topic_model.hierarchical_topics(docs)
print(topic_model.get_topic_tree(hierarchical_topics))

100%|██████████| 28/28 [00:00<00:00, 810.05it/s]

.
├─atlas_data_mongodb_search_databricks
│    ├─atlas_data_mongodb_search_databricks
│    │    ├─atlas_data_mongodb_search_databricks
│    │    │    ├─■──cli_atlas_install_telemetry_command ── Topic: 17
│    │    │    └─■──data_mongodb_atlas_databricks_search ── Topic: 0
│    │    └─dimension_metastore_model_privilege_attribution
│    │         ├─■──metastore_privilege_securable_catalog_principal ── Topic: 16
│    │         └─■──dimension_attribution_metric_model_item ── Topic: 18
│    └─diabetes_puede_gov_cuerpo_salud
│         ├─diabetes_puede_cuerpo_bacterias_enfermedad
│         │    ├─puede_enfermedad_bacterias_pueden_infecciones
│         │    │    ├─■──covid_19_coronavirus_virus_post ── Topic: 24
│         │    │    └─■──puede_bacterias_pueden_infecciones_enfermedad ── Topic: 6
│         │    └─diabetes_glucosa_sangre_adults_type
│         │         ├─■──diabetes_glucosa_adults_type_sangre ── Topic: 11
│         │         └─■──nanoparticles_anticuerpos_immune_antibodies_cells ──

## Zero-shot Topic Modeling

In [9]:
from bertopic.representation import KeyBERTInspired
umap_model = UMAP(random_state=42)
zeroshot_topics = ['Business', 'Finance', 'Law', 'Technology','Science', 'Health']

topic_model2 = BERTopic(
    umap_model=umap_model,
    embedding_model="thenlper/gte-small", #recompute embeddings, throws error when using pre-computed ones
    min_topic_size=15,
    zeroshot_topic_list=zeroshot_topics,
    zeroshot_min_similarity=.75,
    representation_model=KeyBERTInspired(),
    vectorizer_model=vectorizer_model,
    nr_topics='auto'
)
topics, _ = topic_model2.fit_transform(docs)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [10]:
topic_model2.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,40,-1_pcbs_carcinogenic_diabetes_thyroid,"[pcbs, carcinogenic, diabetes, thyroid, tcdd, ...",[Complex technical mixtures of polychlorinated...
1,0,1086,Science,"[noaa, education, educational, environmental, ...","[The 21st CCLC program, the largest out-of-sch..."
2,1,865,Technology,"[databricks, mongodb, database, sql, data, atl...",['Databricks Apps lets developers create secur...
3,2,817,Finance,"[banking, banks, bank, deposits, savings, acco...",[Online banks offer many traditional bank serv...
4,3,665,Law,"[jurisdiction, judicial, court, courts, law, c...",[Most of the cases that the United States Supr...
5,4,626,Health,"[antioxidants, pregnancy, placebo, health, inm...",[Main results:\n \n \n We inc...
6,5,444,Business,"[databricks, data, database, tables, databases...",[This get started article walks you through us...


In [11]:
topic_model2.get_document_info(docs)

,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,"A.D.A.M., Inc. estÃ¡ acreditada por la URAC, t...",4,Health,"[antioxidants, pregnancy, placebo, health, inm...",[Main results:\n \n \n We inc...,antioxidants - pregnancy - placebo - health - ...,0.907601,False
1,"""This dataset for NOAA's Science On a Sphere d...",0,Science,"[noaa, education, educational, environmental, ...","[The 21st CCLC program, the largest out-of-sch...",noaa - education - educational - environmental...,0.922258,False
2,"Dimension items include ""Foreground"" and ""Back...",1,Technology,"[databricks, mongodb, database, sql, data, atl...",['Databricks Apps lets developers create secur...,databricks - mongodb - database - sql - data -...,0.876961,False
3,"Atlas Search runs a new process, called mongot...",1,Technology,"[databricks, mongodb, database, sql, data, atl...",['Databricks Apps lets developers create secur...,databricks - mongodb - database - sql - data -...,0.913095,False
4,The business implications are stark. In a surv...,5,Business,"[databricks, data, database, tables, databases...",[This get started article walks you through us...,databricks - data - database - tables - databa...,0.900787,False
...,...,...,...,...,...,...,...,...
4538,"La glucosa en la sangre, o azÃºcar en la sangr...",4,Health,"[antioxidants, pregnancy, placebo, health, inm...",[Main results:\n \n \n We inc...,antioxidants - pregnancy - placebo - health - ...,0.896686,False
4539,"Currently, Atlas Charts does not have any expe...",0,Science,"[noaa, education, educational, environmental, ...","[The 21st CCLC program, the largest out-of-sch...",noaa - education - educational - environmental...,0.878375,False
4540,'The National Oceanic and Atmospheric Administ...,0,Science,"[noaa, education, educational, environmental, ...","[The 21st CCLC program, the largest out-of-sch...",noaa - education - educational - environmental...,0.903129,False
4541,We are improving Natural Language mode to redu...,0,Science,"[noaa, education, educational, environmental, ...","[The 21st CCLC program, the largest out-of-sch...",noaa - education - educational - environmental...,0.868671,False


In [12]:
topic_model2.visualize_hierarchy(custom_labels=True)

In [13]:
topic_model2.get_document_info(docs).to_csv('topic_assignments.csv', index = False)